# 01 - Quick Start and Core Workflow

> **One line of code, ten thousand rows of data. Zero-config smart generation, AI-driven precise tuning.**

## Scenario Navigation

| My Need | Recommended Notebook | Core API |
|----------|---------------|----------|
| Quickly generate test data | **01 - Quick Start** ← you are here | `fill()` |
| Customize data type per column | 02 - Column Mapping | `columns={}` |
| Choose data generation engine | 03 - Generators and Providers | `provider=` |
| Multi-table relations, FK integrity | 04 - Database and Multi-table | `connect()` + `fill_from_config()` |
| Derivation between columns | 05 - Expressions and Constraints | `derive_from` + `expression` |
| YAML config-driven, batch generation | 06 - Config and Transform | `fill_from_config()` |
| AI auto-generates config | 07 - AI Smart Config | `sqlseed-ai` plugin |
| AI assistant operates database | 08 - MCP Server | `mcp-server-sqlseed` |
| Custom plugin extensions | 09 - Plugins and Hooks | `pluggy` |
| Command-line operations | 10 - CLI Reference | `sqlseed` CLI |

## What You Will Learn

- One-line fill: the power of `sqlseed.fill()`
- Zero-config smart inference: how sqlseed auto-selects generators
- Preview data: `sqlseed.preview()` without writing to DB
- Context manager: `sqlseed.connect()` for fine-grained control
- Core parameters: count, provider, seed, batch_size, enrich

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| **→ 01** | **Quick Start and Core Workflow** | **Orchestrator** | **None** |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Core Orchestration | `src/sqlseed/core/orchestrator.py` | `DataOrchestrator.fill_table()` |

> Corresponding architecture diagram: [§2 Core Orchestration Flow (fill_table execution chain)](../docs/architecture.zh-CN.md#2-核心编排流程fill_table-执行链路)

## 1. One Line of Code, Ten Thousand Rows — The Power of sqlseed

Core philosophy of sqlseed: **generate massive test data with a single line of code**. No need to write generation scripts or maintain SQL fixtures — sqlseed auto-infers table structure, intelligently selects generation strategies, and streams data into the database.

```python
result = sqlseed.fill("app.db", table="users", count=100_000)
```

In [19]:
# Fill parent table (organizations) first, then child table (members)
# sqlseed needs parent table data to resolve foreign key references
fill(str(db_path), table="organizations", count=5)

# One line of code, generate 100 member rows
result = fill(str(db_path), table="members", count=100)
print(result)
# → GenerationResult(table=members, count=100, elapsed=..., speed=...)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/100 [00:00<?, ?it/s]

GenerationResult(table=members, count=100, elapsed=0.05s, speed=1897.96 rows/s)


With just this one line — sqlseed accomplished:

1. **Read schema** — auto-detect columns, types, constraints of `members` table
2. **Smart mapping** — `name` → real name, `email` → email address, `member_no` → unique ID
3. **Stream write** — batch insert 100 rows, auto-handle UNIQUE constraints
4. **Return result** — `GenerationResult` contains row count, elapsed time, speed, etc.

## 2. Inspect Database Structure

Before generating data, let's see what tables and columns the database has.

In [20]:
import sqlite3

conn = sqlite3.connect(str(db_path))
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
print(f"Tables ({len(tables)}): {tables}")

for table in tables:
    cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
    col_names = [c[1] for c in cols]
    print(f"  {table}: {col_names}")

conn.close()

Tables (9): ['attachments', 'members', 'organizations', 'projects', 'reviews', 'sqlite_sequence', 'tags', 'task_tags', 'tasks']
  attachments: ['attachment_id', 'task_id', 'file_name', 'file_data', 'file_size', 'uploaded_at']
  members: ['member_id', 'member_no', 'name', 'email', 'phone', 'org_code', 'is_active', 'balance', 'avatar', 'registered_at', 'address']
  organizations: ['org_code', 'name', 'parent_code', 'description', 'is_active', 'member_count', 'created_at']
  projects: ['project_id', 'project_no', 'short_code', 'name', 'org_code', 'budget', 'task_count', 'is_public', 'is_archived', 'created_at', 'description']
  reviews: ['review_id', 'task_id', 'member_id', 'rating', 'content', 'created_at']
  sqlite_sequence: ['name', 'seq']
  tags: ['tag_id', 'name', 'color', 'usage_count']
  task_tags: ['task_id', 'tag_id']
  tasks: ['task_id', 'project_id', 'assignee_id', 'title', 'priority', 'status', 'is_completed', 'comment_count', 'estimated_hours', 'due_at', 'completed_at', 'crea

## 3. Zero-Config Fill in Detail

Behind the single `fill()` line above, sqlseed does a lot of work. Let's look in detail:

In [21]:
result = fill(str(db_path), table="members", count=10)
print(result)

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

GenerationResult(table=members, count=10, elapsed=0.03s, speed=318.32 rows/s)


### GenerationResult in Detail

The returned `GenerationResult` contains rich execution info:

In [22]:
print(f"Table name: {result.table_name}")
print(f"Rows inserted: {result.count}")
print(f"Elapsed: {result.elapsed:.3f}s")
print(f"Speed: {result.rows_per_second:.2f} rows/s")
print(f"Batch count: {result.batch_count}")
print(f"Errors: {result.errors}")

Table name: members
Rows inserted: 10
Elapsed: 0.031s
Speed: 318.32 rows/s
Batch count: 10
Errors: []


### View Generated Data

In [23]:
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT member_id, member_no, name, email, org_code FROM members LIMIT 5").fetchall()

# Tabular output
print(f"{'ID':>4s}  {'member_no':<20s}  {'name':<20s}  {'email':<30s}  {'org_code':<10s}")
print('-' * 90)
for row in rows:
    print(f"{row[0]:>4d}  {row[1]:<20s}  {row[2]:<20s}  {row[3]:<30s}  {row[4]:<10s}")
conn.close()

  ID  member_no             name                  email                           org_code  
------------------------------------------------------------------------------------------
   1  nPmDMlFXPgF4ppH       Clementina Kemp       likely1848@protonmail.com       4CPvyCA   
   2  IB7jO7eWeiHlu         Lashawna Henson       hugo2033@example.org            UmTrNDj   
   3  BVUh8MSe              Vannesa Lane          cgi2077@yandex.com              sr7EGr7   
   4  btjym                 Walter Alford         yes1828@example.org             UmTrNDj   
   5  JcEVy2ajAR24          Cesar Wong            surfing2038@gmail.com           YdcroT    


## 4. Preview Data (Without Writing to DB)

`sqlseed.preview()` generates data but **does not write to the database**, suitable for debugging and verifying mapping results.

In [24]:
# Fill projects table first (requires organizations to have data)
fill(str(db_path), table="projects", count=5)

# View generated project info
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name, org_code FROM projects").fetchall()
print(f"{'project_no':<12s}  {'short_code':<8s}  {'name':<25s}  {'org_code':<10s}")
print('-' * 58)
for row in rows:
    print(f"{row[0]:<12s}  {str(row[1] or ''):<8s}  {row[2]:<25s}  {row[3]:<10s}")
conn.close()

# preview generates new data (no write), can use columns to override generation strategy
print("\npreview generates new data (columns override):")
rows = preview(str(db_path), table="projects", count=3,
               columns={"project_no": {"type": "pattern", "regex": PRJ_PATTERN},
                        "name": {"type": "company"}})
for row in rows:
    print(f"  {row.get('project_no'):<12s}  {row.get('name')}")

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no    short_code  name                       org_code  
----------------------------------------------------------
iMTTLoG0vOXYf  FO1wmL9F3  Lesley Stuart              YdcroT    
gANUHUxQNmxluGNbAf  QOKb4z    Marcos Stone               4CPvyCA   
cQ7WfHRj2z7   nZRqs2VC  Ron Nielsen                UmTrNDj   
ngeQ          evtGFP    Felipe Burnett             TFCTkQU9LZq7
hukbS rYnIuCFR cGKn  lJFdWFSeP  Philip Sweet               UmTrNDj   

preview generates new data (columns override):
  PRJ-132130    American Eagle Outfitters
  PRJ-491990    AirTran Holdings
  PRJ-241327    Gamma Gas


## 5. Context Manager

`sqlseed.connect()` returns a `DataOrchestrator` context manager, suitable for scenarios requiring multiple fills. It auto-cleans resources on exit.

In [25]:
with connect(str(db_path), provider="mimesis", locale="en") as orch:
    r1 = orch.fill_table("tags", count=10)
    r2 = orch.fill_table("tasks", count=50)
    print(f"Tags: {r1.count} rows in {r1.elapsed:.3f}s")
    print(f"Tasks: {r2.count} rows in {r2.elapsed:.3f}s")

Generating tags:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/50 [00:00<?, ?it/s]

Tags: 10 rows in 0.033s
Tasks: 50 rows in 0.036s


## 6. Core Parameters in Detail

### 6.1 count — Number of Rows

Controls the volume of data generated. For columns with UNIQUE constraints, sqlseed auto-backtracks to ensure no duplicates.

In [26]:
import sqlite3

conn = sqlite3.connect(str(db_path))
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]

result = fill(str(db_path), table="organizations", count=3,
              columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                       "parent_code": {"type": "choice", "choices": existing_codes}})
print(f"Generated {result.count} organizations ({result.elapsed:.3f}s, {result.rows_per_second:.0f} rows/s)")

rows = conn.execute("SELECT org_code, name, parent_code FROM organizations").fetchall()
print(f"\n{'org_code':<12s}  {'name':<25s}  {'parent_code':<12s}")
print('-' * 52)
for row in rows:
    parent = row[2] or '(root)'
    print(f"{row[0]:<12s}  {row[1]:<25s}  {parent:<12s}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generated 3 organizations (0.022s, 134 rows/s)

org_code      name                       parent_code 
----------------------------------------------------
UmTrNDj       Samira Mayer               673557      
4CPvyCA       Lisandra Snider            56685       
YdcroT        Tamiko Joyner              77537       
sr7EGr7       Darryl Townsend            299321      
TFCTkQU9LZq7  Bibi West                  491709      
ORG-7773      Lee Serrano                4CPvyCA     
ORG-1238      Hiedi Lamb                 TFCTkQU9LZq7
ORG-1246      Herma Lynch                TFCTkQU9LZq7


### 6.2 provider — Data Provider

sqlseed supports three Providers, descending by richness:

| Provider | Dependency | Quality | Speed |
|----------|------|----------|------|
| `mimesis` | mimesis | Highest (localized, semantically rich) | Fast |
| `faker` | faker | High (many methods, large community) | Medium |
| `base` | None | Basic (random strings/numbers) | Fastest |

In [27]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 60
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")


  Provider: base
  name=Joseph Robinson      email=margaret.martinez185@sample.dev
  phone=778-269-1431         address=2059 Washington Ave, Salem, PA
  name=David Rodriguez      email=deborah.ramirez925@mail.net   
  phone=990-912-2228         address=9633 Park Rd, Georgetown, CA

  Provider: faker
  name=Erin Mcclain         email=jonathangarrett@example.com   
  phone=224-573-7761x513     address=9342 Hill Skyway Apt. 282, South Jamesbu
  name=Jenna Heath          email=loricastillo@example.com      
  phone=001-830-771-7590x268 address=22765 Christopher Crossing, Port Ronaldb

  Provider: mimesis
  name=Emely William        email=shades1809@example.com        
  phone=+1-401-424-2727      address=46 Service Avenue
  name=Mendy Henson         email=wage1919@yandex.com           
  phone=+1-405-418-2605      address=1116 Albatross Canyon


### 6.3 seed — Reproducibility

Setting the same seed ensures the same data is generated each time, suitable for testing and debugging.

In [28]:
rows_a = preview(str(db_path), table="members", count=3, seed=42)
rows_b = preview(str(db_path), table="members", count=3, seed=42)
rows_c = preview(str(db_path), table="members", count=3, seed=99)

names_a = [r["name"] for r in rows_a]
names_b = [r["name"] for r in rows_b]
names_c = [r["name"] for r in rows_c]

print(f"seed=42 (run 1): {names_a}")
print(f"seed=42 (run 2): {names_b}")
print(f"seed=99:        {names_c}")
print(f"\nseed=42 reproducible: {names_a == names_b}")
print(f"Different seeds:  {names_a != names_c}")

seed=42 (run 1): ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
seed=42 (run 2): ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
seed=99:        ['Myung Cote', 'Lyle Shields', 'Nicky Craig']

seed=42 reproducible: True
Different seeds:  True


### 6.4 batch_size — Batch Write Size

Controls the number of rows written per batch. Larger batch_size improves write performance but uses more memory.

- Default: 5000
- Small data volume (<1000 rows): batch_size has little impact
- Large data volume (>10K rows): increasing batch_size can improve speed

In [29]:
# batch_size controls rows per batch, default 5000
# Internally auto-adjusted for smooth progress bar display
r = fill(str(db_path), table="members", count=50)
print(f"50 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

r = fill(str(db_path), table="members", count=500)
print(f"500 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

50 rows: 0.035s, 1440 rows/s


Generating members:   0%|          | 0/500 [00:00<?, ?it/s]

500 rows: 0.117s, 4272 rows/s


### 6.5 enrich — Smart Enrichment Mode

When the database already has some data, `enrich=True` makes sqlseed analyze existing data patterns (e.g., enum values, value ranges) and keep them consistent when generating new data.

See architecture.md §2 enrich flow

In [30]:
import sqlite3

# enrich demo: first check existing data in organizations table

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
orgs = conn.execute("SELECT org_code, name FROM organizations").fetchall()
print(f"Existing data: {before} organizations")
for r in orgs:
    print(f"  {r[0]:<12s} {r[1]}")

# Get existing org_code as candidate values for parent_code
existing_codes = [r[0] for r in orgs]

# enrich=True: analyze existing data patterns, new data stays consistent
r2 = fill(str(db_path), table="organizations", count=5, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                   "parent_code": {"type": "choice", "choices": existing_codes}})
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 5").fetchall()
print(f"\nenrich added {r2.count} ({before} -> {after}):")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()


Existing data: 8 organizations
  UmTrNDj      Samira Mayer
  4CPvyCA      Lisandra Snider
  YdcroT       Tamiko Joyner
  sr7EGr7      Darryl Townsend
  TFCTkQU9LZq7 Bibi West
  ORG-7773     Lee Serrano
  ORG-1238     Hiedi Lamb
  ORG-1246     Herma Lynch


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]


enrich added 5 (8 -> 13):
  ORG-8195     Kelsie Mayer           parent=sr7EGr7
  ORG-4496     Loma Woods             parent=YdcroT
  ORG-4665     Brendon Dodson         parent=UmTrNDj
  ORG-3885     Cassy Stevens          parent=YdcroT
  ORG-2094     Erik Valenzuela        parent=YdcroT


### 6.6 clear_before — Clear Then Fill

Clears existing data in the table before filling. Note: if there are foreign key references, fill the referenced table first.

In [31]:
result = fill(str(db_path), table="tags", count=8, clear_before=True)
print(f"Cleared and refilled: {result.count} rows")

conn = sqlite3.connect(str(db_path))
count = conn.execute("SELECT COUNT(*) FROM tags").fetchone()[0]
print(f"Current row count: {count}")
conn.close()

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

Cleared and refilled: 8 rows
Current row count: 8


## 7. Custom Column Mapping

Use the `columns` parameter to override auto-inferred column mapping:

In [32]:
result = fill(
    str(db_path),
    table="members",
    count=5,
    columns={
        "name": {"type": "name"},               # real name
        "email": {"type": "email"},             # email address
        "phone": {"type": "phone"},             # phone number
        "balance": {"type": "float", "min_value": 100.0, "max_value": 500.0},  # specified range
        "is_active": {"type": "boolean"},       # boolean
    },
)
print(f"Custom mapping: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, email, phone, balance, is_active FROM members ORDER BY member_id DESC LIMIT 5").fetchall()  # noqa: E501
print(f"\n{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'balance':>8s}  {'active':>6s}")
print('-' * 85)
for row in rows:
    print(f"{row[0]:<18s}  {row[1]:<28s}  {row[2]:<18s}  {row[3]:>8.2f}  {row[4]!s:>6s}")
conn.close()

Generating members:   0%|          | 0/5 [00:00<?, ?it/s]

Custom mapping: 5 rows

name                email                         phone                balance  active
-------------------------------------------------------------------------------------
Raymonde Small      benjamin2050@yandex.com       +13097970592          317.87       0
Ashlea Calhoun      purpose1916@yandex.com        +1-480-695-0643       188.75       1
Nakisha Hansen      ceramic1929@yandex.com        +15151737329          184.20       1
Doreatha Diaz       treat1884@yandex.com          +1-508-366-9905       437.64       0
Bernardina Hernandez  true2061@example.org          +1-727-572-3530       304.01       0


## 🎯 enrich Mode in Detail

When `enrich=True`, sqlseed auto-detects **enum columns** (e.g., `status`, `*_type`, `is_*`, etc.), and even if these columns have DEFAULT values or are nullable, it generates meaningful enum values instead of skipping them.

EnrichmentEngine uses 19 enum column name patterns and cardinality ratio calculation to identify enum columns. See [02-column-mapping](02-column-mapping.ipynb).

In [33]:
import sqlite3

# enrich mode in detail: demo on organizations table
# First fill baseline data
fill(str(db_path), table="organizations", count=3, seed=42,
     columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"}})

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]
print(f"Before enrich: {before} organizations")

# enrich adds organizations, auto-keeping existing data patterns
r2 = fill(str(db_path), table="organizations", count=3, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"},
                   "parent_code": {"type": "choice", "choices": existing_codes}})
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 3").fetchall()
print(f"After enrich: {after} organizations (added {r2.count})")
print("\nenrich added:")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Before enrich: 16 organizations


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

After enrich: 19 organizations (added 3)

enrich added:
  ENR-7457     Petronila Workman      parent=YdcroT
  ENR-3083     Denyse Hampton         parent=ORG-2094
  ENR-2881     Colby Valenzuela       parent=TFCTkQU9LZq7


## 📋 fill_from_config Getting Started

Simplest YAML config + one `fill_from_config()` call to batch-fill multiple tables. See [06-config-deep-dive](06-config-deep-dive.ipynb).

In [34]:
from pathlib import Path
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

simple_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3),
    ]
)
config_path = Path("_quickstart_config.yaml")
save_config(simple_config, str(config_path))

results = fill_from_config(str(config_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

  organizations: 3 rows in 0.022s


## ⚠️ Error Handling

Common errors and how to handle them:

In [35]:
r1 = fill(str(db_path), table="nonexistent_table", count=1)
if r1.errors:
    print(f"Table not found (internally caught): {r1.errors[0]}")

try:
    fill(str(db_path), table="organizations", count=-1)
except ValueError as e:
    print(f"Parameter error (exception raised): {e}")

try:
    fill(str(db_path), table="; DROP TABLE organizations; --", count=1)
except ValueError as e:
    print(f"SQL injection protection: {e}")

2026-05-05T23:06:42.363028Z [error    ] Failed to fill table           error=OperationalError('no such table: nonexistent_table') table_name=nonexistent_table


Table not found (internally caught): no such table: nonexistent_table
Parameter error (exception raised): count must be greater than 0, got -1


2026-05-05T23:06:42.451253Z [warning  ] Table name '; DROP TABLE organizations; --' contains special characters and will be quoted


SQL injection protection: SQL identifier '; DROP TABLE organizations; --' contains dangerous characters and is rejected


## 8. Summary

| API | Purpose | Writes to DB |
|-----|------|:----------:|
| `fill()` | One-line fill | ✅ |
| `preview()` | Preview without write | ❌ |
| `connect()` | Context manager | ✅ |
| `fill_from_config()` | YAML/JSON batch fill | ✅ |

| Parameter | Default | Description |
|------|--------|------|
| `count` | 1000 | Number of rows |
| `provider` | mimesis | Data provider (mimesis/faker/base) |
| `seed` | None | Random seed, reproducible when set |
| `batch_size` | 5000 | Batch write size |
| `enrich` | False | Smart enrichment mode |
| `clear_before` | False | Clear then fill |
| `locale` | en_US | Locale setting |

**Next**: [02-column-mapping.ipynb](02-column-mapping.ipynb) — Deep dive into the 9-level strategy chain

In [36]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
